# Hyperparameter tuning

### 01. Import libraries

In [1]:
import os
import copy
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision import transforms
from torchvision import models
from torch.optim import AdamW, SGD
from torch.optim.lr_scheduler import CosineAnnealingLR, StepLR

### 02. Path and hyperparameter configuration

In [2]:
DATASET_DIR = Path('../dataset')
TRAIN_DIR = DATASET_DIR / 'train'
VAL_DIR = DATASET_DIR / 'validation'
MODEL_PATH = Path('../models/best_convnext_tiny_finetuned.pth')

IMAGE_SIZE = 224
NUM_CLASSES = 46
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
BATCH_SIZE = 32

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

### 03. Create datasets and dataloaders

In [3]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

validation_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [4]:
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
validation_dataset = datasets.ImageFolder(VAL_DIR, transform=validation_transform)

In [5]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

### 04. Load fine-tuned model

In [6]:
model = models.convnext_tiny(weights=None)
model.classifier[2] = nn.Linear(model.classifier[2].in_features, NUM_CLASSES)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model = model.to(DEVICE)
print('Fine-tuned model loaded successfully.')

Fine-tuned model loaded successfully.


### 05. Define experiments

In [18]:
EXPERIMENTS = [
    {
        'name': 'Experiment_1',
        'learning_rate': 1e-6,
        'weight_decay': 1e-4
    },
    {
        'name': 'Experiment_2',
        'learning_rate': 5e-7,
        'weight_decay': 1e-4
    },
    {
        'name': 'Experiment_3',
        'learning_rate': 1e-6,
        'weight_decay': 1e-5
    }
]

NUM_EPOCHS = 5
GLOBAL_BEST_ACCURACY = 0.0
BEST_MODEL_PATH = '../models/best_convnext_tiny_hyperparameter_tuned.pth'

### 06. Function for one experiment

In [19]:
def run_experiment(experiment):

    global GLOBAL_BEST_ACCURACY

    print(experiment['name'])
    print(f'Learning Rate: {experiment['learning_rate']}')
    print(f'Weight Decay: {experiment['weight_decay']}')

    experiment_model = copy.deepcopy(model)
    experiment_model = experiment_model.to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(experiment_model.parameters(), lr=experiment['learning_rate'], weight_decay=experiment['weight_decay'])
    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    best_val_accuracy = 0
    best_epoch = 0

    start_time = time.time()

    for epoch in range(NUM_EPOCHS):
        experiment_model.train()
        train_correct = 0
        train_total = 0
        train_loss = 0

        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = experiment_model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            predictions = torch.argmax(outputs, dim=1)
            train_correct += (predictions == labels).sum().item()
            train_total += labels.size(0)

        train_accuracy = train_correct / train_total

        experiment_model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0

        with torch.no_grad():
            for images, labels in validation_loader:
                images = images.to(DEVICE)
                labels = labels.to(DEVICE)
                outputs = experiment_model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                predictions = torch.argmax(outputs, dim=1)
                val_correct += (predictions == labels).sum().item()
                val_total += labels.size(0)

        val_accuracy = val_correct / val_total

        scheduler.step()

        print(
            f'Epoch [{epoch+1}/{NUM_EPOCHS}] '
            f'| Train Acc: {train_accuracy:.4f} '
            f'| Val Acc: {val_accuracy:.4f}'
        )

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_epoch = epoch + 1

        if val_accuracy > GLOBAL_BEST_ACCURACY:
            GLOBAL_BEST_ACCURACY = val_accuracy
            torch.save(experiment_model.state_dict(), BEST_MODEL_PATH)
            print('GLOBAL BEST MODEL SAVED!')

    elapsed = time.time() - start_time

    print(f'\nFinished {experiment['name']}')
    print(f'Best Validation Accuracy: {best_val_accuracy:.4f}')

    return {
        'Experiment': experiment['name'],
        'Learning Rate': experiment['learning_rate'],
        'Weight Decay': experiment['weight_decay'],
        'Best Epoch': best_epoch,
        'Validation Accuracy': best_val_accuracy,
        'Training Time (min)': elapsed / 60
    }

In [20]:
results = []

for experiment in EXPERIMENTS:
    result = run_experiment(experiment)
    results.append(result)

Experiment_1
Learning Rate: 1e-06
Weight Decay: 0.0001
Epoch [1/5] | Train Acc: 0.9721 | Val Acc: 0.8643
GLOBAL BEST MODEL SAVED!
Epoch [2/5] | Train Acc: 0.9760 | Val Acc: 0.8661
GLOBAL BEST MODEL SAVED!
Epoch [3/5] | Train Acc: 0.9772 | Val Acc: 0.8663
GLOBAL BEST MODEL SAVED!
Epoch [4/5] | Train Acc: 0.9788 | Val Acc: 0.8654
Epoch [5/5] | Train Acc: 0.9795 | Val Acc: 0.8655

Finished Experiment_1
Best Validation Accuracy: 0.8663
Experiment_2
Learning Rate: 5e-07
Weight Decay: 0.0001
Epoch [1/5] | Train Acc: 0.9716 | Val Acc: 0.8639
Epoch [2/5] | Train Acc: 0.9735 | Val Acc: 0.8648
Epoch [3/5] | Train Acc: 0.9738 | Val Acc: 0.8664
GLOBAL BEST MODEL SAVED!
Epoch [4/5] | Train Acc: 0.9742 | Val Acc: 0.8663
Epoch [5/5] | Train Acc: 0.9757 | Val Acc: 0.8665
GLOBAL BEST MODEL SAVED!

Finished Experiment_2
Best Validation Accuracy: 0.8665
Experiment_3
Learning Rate: 1e-06
Weight Decay: 1e-05
Epoch [1/5] | Train Acc: 0.9716 | Val Acc: 0.8646
Epoch [2/5] | Train Acc: 0.9761 | Val Acc: 0.8655

### 07. Experiments results

In [21]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df.sort_values(by='Validation Accuracy', ascending=False)

,Experiment,Learning Rate,Weight Decay,Best Epoch,Validation Accuracy,Training Time (min)
1,Experiment_2,5.000000e-07,0.00010,5,0.866535,720.771740
0,Experiment_1,1.000000e-06,0.00010,3,0.866254,719.344545
2,Experiment_3,1.000000e-06,0.00001,4,0.866254,718.776619
